## Graph construction

- Construct NN-graphs w/ $G(V,E)$ with nodes $v_i \in V$ as DAPI coordinates & edges $(e_i, e_j) \in E$ as affinity btw adjacent nodes

- Graph comparisons btw L-slices w/ H&E channels, CyIF channels & joint channels

In [ ]:
import os
import gc
import sys
import pickle

import numpy as np
import cupy as cp
import pandas as pd

import networkx as nx
import tifffile

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse

import torch
import torch.nn as nn

from scipy.sparse import csr_matrix
from skimage.transform import rescale
from torch_geometric.nn import VGAE, GCNConv
from torch_geometric import utils as pyg_utils
from ml_collections import ConfigDict

import torch
torch.manual_seed(42)

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
from cellpose import utils as cp_utils
from cellpose import plot as cp_plot
# from cellpose.models import CellposeModel

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
import IO, utils

### Segmentation & Graph Construction

#### CyIF channels

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CellposeModel(
    gpu=device,
    model_type='nuclei',
    # pretrained_model=pretrained_model.value
)

In [ ]:
#  Load dataset
data_path = '../data/cycif/zonation_hires/'
filenames = [f for f in sorted(os.listdir(data_path))
             if f[-3:] == 'tif']
dapis = [tifffile.imread(os.path.join(data_path, f))[0] for f in filenames]
gc.collect()


In [ ]:
out_path = '../data/cycif/segment/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

for f, img in zip(filenames, dapis):
    print('Segmentation on image {}...'.format(f))
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min())

    mask = model.eval(
        img,
        diameter=8,
        tile=True,
        channels=[0, 0],
        flow_threshold=1,
        min_size=50
    )[0]
    torch.cuda.empty_cache()

    tifffile.imwrite(os.path.join(out_path, f.split('.')[0] + '_mask.tif'), mask)
    

Construct k-NN graphs:
- Try not subsampling graph upon initialization
- Graph aggregation w/ convolution operators

In [ ]:
img_path = '../data/cycif/zonation_hires/'
mask_path = '../data/cycif/segment/'

filenames = [f for f in sorted(os.listdir(img_path))
             if f[-3:] == 'tif']
dapis = [tifffile.imread(os.path.join(img_path, f))[0]
        for f in filenames]
masks = [tifffile.imread(os.path.join(mask_path, f.split('.')[0]+'_mask.tif'))
         for f in filenames]


In [ ]:
def get_centroids(mask):
    mempool = cp.get_default_memory_pool()

    mask_cp = cp.array(mask)
    coords = []
    labels = cp.unique(mask_cp)[1:]
    for label in labels:
        pts_y, pts_x = cp.nonzero(mask_cp == label)
        cy = cp.round(pts_y.mean()).astype(cp.uint32)
        cx = cp.round(pts_x.mean()).astype(cp.uint32)
        coords.append([int(cy.get()), int(cx.get())])
        mempool.free_all_blocks()    

    return np.array(coords)


def get_coords_from_graph(G, node_list=None):
    try:
        nodes = G.nodes if node_list is None else node_list
        return np.array([
            [G.nodes[n]['pos'][1], G.nodes[n]['pos'][0]]
             for n in nodes
        ])
    except KeyError as e:
        print(e, 'Please assign `pos` to graph nodes first')


def construct_graph(coords, k=5, r=30):
    G = nx.Graph()
    nbrs = NearestNeighbors(n_neighbors=k+1, metric='euclidean').fit(coords)
    distances, nn_indices = nbrs.kneighbors(coords)
    distances, nn_indices = distances[:, 1:], nn_indices[:, 1:]  # Skip self

    # Construct Graph w/ all nodes
    for i, (y, x) in enumerate(coords):
        G.add_node(i, pos=(x, y))  # XY-based coords

    # Add edges within `r` resolution
    for i in range(len(coords)):
        for j, distance in zip(nn_indices[i], distances[i]):
            if i != j and (r == np.inf or distance <= r):
                G.add_edge(i, j, weight=1/distance)

    return G


def sample_nodes(G, k=5, r=np.inf, 
                 sample_ratio=0.1, res=0.5):
    partition = nx.community.louvain_communities(G, resolution=res)
    sampled_nodes = []
    for nodes in partition:
        sample_size = np.ceil(len(nodes)*sample_ratio).astype(np.uint8)
        sampled_nodes.extend(np.random.choice(list(nodes), size=sample_size, replace=False))
    coords = get_coords_from_graph(G, sampled_nodes)
    G_new = construct_graph(coords, k=k, r=r)
    return G_new


def construct_feature_matrix(img, coords, r=4):
    assert img.ndim == 3,\
        "Image dimension needs to be (C, Y, X)"
    
    n_nodes, n_chans = coords.shape[0], img.shape[0]
    ydim, xdim = img.shape[1:]
    features = np.zeros((n_nodes, n_chans))
    for i, (y, x) in enumerate(coords):
        
        indices = (slice(0, n_chans), 
                   slice(max(0, y-r), min(ydim-1, y+r)), 
                   slice(max(0, x-r), min(xdim-1, x+r)))
        features[i] = img[indices].sum((1, 2))

    return features


def disp_network(img, G, figsize=None,
                 node_size=5, edge_width=1, fontsize=20,
                 title=None):
    pos = {n: G.nodes[n]['pos'] for n in G.nodes}
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img, cmap='magma', alpha=0.5)
    ax.axis('off')
    nx.draw_networkx_nodes(G, pos, node_color='yellow', node_size=node_size, ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='lime', width=edge_width, ax=ax)
    plt.tight_layout()
    plt.title(title, fontsize=fontsize)


def generate_random_colors(n):
    random_colors = []
    for _ in range(n):
        # Generate random RGB values
        r = np.random.randint(0, 255)
        g = np.random.randint(0, 255)
        b = np.random.randint(0, 255)
        # Convert RGB values to hexadecimal color code
        color_code = "#{:02x}{:02x}{:02x}".format(r, g, b)
        random_colors.append(color_code)
    return random_colors


def disp_graph_overlaps(Gs, labels, figsize,
                        node_size=5, edge_width=1,
                        title=None):
    colors = generate_random_colors(len(Gs))
    fig, ax = plt.subplots(figsize=figsize)
    for c, lbl, G in zip(colors, labels, Gs):
        pos = nx.get_node_attributes(G, 'pos')
        nx.draw_networkx_nodes(G, pos, node_color=c, node_size=node_size, label=lbl, ax=ax)
        nx.draw_networkx_edges(G, pos, edge_color=c, width=edge_width, ax=ax)

    plt.legend()
    plt.tight_layout()
    plt.title(title, fontsize=20)
    plt.show() 


def disp_network_embedding(img, G, embedding, figsize=None, 
                           alpha=0.5, img_vmax=64, 
                           node_size=5, edge_width=1,
                           fontsize=20, title=None):
    # coords = get_coords_from_graph(G)
    pos = {n: G.nodes[n]['pos'] for n in G.nodes}
    
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img, cmap='magma', alpha=alpha, vmax=img_vmax)
    # im = ax.scatter(coords[:, 1], coords[:, 0], s=node_size, c=embedding, cmap='jet')
    im = nx.draw_networkx_nodes(G, pos, node_color=embedding, node_size=node_size, cmap='jet', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', width=edge_width, ax=ax)
    ax.axis('off')    
    plt.colorbar(im, ax=ax, fraction=0.02, pad=1e-4)
    plt.tight_layout()
    plt.title(title, fontsize=fontsize, y=0.95)
    plt.show()


def disp_corr_features(features, labels=None, titles=None, ncols=4):
    n_slices = len(features)
    nrows = n_slices // ncols if n_slices % ncols == 0 else n_slices // ncols + 1

    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 3*nrows), dpi=200)
    for r in range(nrows):
        for c in range(ncols):
            if idx >= n_slices:
                axes[r, c].axis('off')
                continue
            corr = np.corrcoef(features[idx].T)
            sns.heatmap(corr, mask=np.triu(corr),
                        xticklabels=labels, yticklabels=labels,
                        vmin=-0.3, vmax=0.3, square=True, 
                        ax=axes[r, c], cmap='coolwarm')
            
            if titles is not None:
                axes[r, c].set_title(titles[idx])
            idx += 1
    plt.tight_layout()
    plt.suptitle('Feature correlations (w/ Graph)', fontsize=30, y=1.02)
    plt.show()
    return None
    

In [ ]:
# Save simplified "meta-graphs" per L-slice
out_path = '../data/cycif/graph/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

# prop_nodes_to_keep = 0.1

for f, mask in zip(filenames, masks):
    print('Computing graph for {}...'.format(f))
    coords = get_centroids(mask)
    # G_raw = construct_graph(coords, k=5, r=30)
    # G = sample_nodes(G_raw, sample_ratio=0.1, k=3)
    
    G = construct_graph(coords, k=3, r=10)
    pickle.dump(G, open(os.path.join(out_path, f.split('.')[0] + '.pkl'), 'wb'))

    del G
    gc.collect()

In [ ]:
Gs = []
for f, dapi in zip(filenames, dapis):
    G =  pickle.load(open(os.path.join(out_path, f.split('.')[0] + '.pkl'), 'rb'))
    Gs.append(G)
    del G, dapi

In [ ]:
L = len(Gs)
n_nodes = [len(G) for G in Gs]

fig, ax = plt.subplots(figsize=(8, 2))
ax.plot(np.arange(L), n_nodes, '.-')
ax.set_xlabel('# section')
ax.set_ylabel(r"$\vert V \vert$")

plt.show()

del L

In [ ]:
disp_network(dapis[0], Gs[0], figsize=(15, 15), node_size=0.2, edge_width=0.2)

In [ ]:
disp_graph_overlaps(Gs[3:6], labels=filenames[3:6], figsize=(8, 8), 
                    node_size=0.5, edge_width=0.1)

Feature matrix extraction

In [ ]:
cyif_chans = [
    'DAPI',
    'GS 647', 
    'CYP3A4', 
    'ASS1 PE',
    'Pan CK',
    'CD31', 
    'CD45', 
    'CD56', 
    'CD68'
]

In [ ]:
img_path = '../data/cycif/zonation_hires/'
graph_path = '../data/cycif/graph/'
mask_path = '../data/cycif/segment/'

filenames = [f for f in sorted(os.listdir(img_path))
             if f[-3:] == 'tif']

# Load images
imgs_rgb = [tifffile.imread(os.path.join(img_path, f))
            for f in filenames]

# Load graphs
Gs = [pickle.load(open(os.path.join(graph_path, f), 'rb'))
      for f in sorted(os.listdir(graph_path))
      if f[-3:] == 'pkl']

# Retrieve graph coords
coords_list = [get_coords_from_graph(G) for G in Gs]

In [ ]:
# Convert channel expressions to [0, 1]
# Covert empty channels to 0s
imgs = []
for img in imgs_rgb:
    img_adj = img.copy().astype(np.float32)
    for i, chan in enumerate(img):
        if chan.max() <= 1:
            img_adj[i] = 0
        else:
            img_adj[i] = (img_adj[i] - img_adj[i].min()) / (img_adj[i].max() - img_adj[i].min())
    imgs.append(img_adj)
del imgs_rgb

In [ ]:
features = [utils.znorm(construct_feature_matrix(img, coords, r=5))
            for img, coords in zip(imgs, coords_list)]

In [ ]:
disp_corr_features(features, labels=cyif_chans, titles=filenames, ncols=6)

In [ ]:
# Save feature 
graph_path = '../data/cycif/graph/'
for f, feature in zip(filenames, features):
    feature_df = pd.DataFrame(feature, columns=cyif_chans)
    feature_df.to_csv(os.path.join(graph_path, f.split('.')[0] + '.csv'), index=True)

#### H&E channels

### VGAE

2-layer dim-reduction, first trained on a single graph

In [ ]:
import vgae, utils
import model_train, model_eval, configs

In [ ]:
from importlib import reload
reload(vgae)
reload(model_train)
reload(model_eval)
reload(configs)

In [ ]:
img_path = '../data/cycif/zonation_hires/'
graph_path = '../data/cycif/graph/'
mask_path = '../data/cycif/segment/'

filenames = [f for f in sorted(os.listdir(img_path))
             if f[-3:] == 'tif']

# Load images
imgs_rgb = [tifffile.imread(os.path.join(img_path, f))
            for f in filenames]

# Load graphs
Gs = [pickle.load(open(os.path.join(graph_path, f), 'rb'))
      for f in sorted(os.listdir(graph_path))
      if f[-3:] == 'pkl']

# Retrieve graph coords
coords_list = [get_coords_from_graph(G) for G in Gs]

In [ ]:
# Model configs & initialization
train_configs = configs.set_train_configs(data_path=None, n_epochs=1000, lr=1e-3)
model_configs = configs.set_model_configs(c_in=9,
                                          c_hidden=4,
                                          c_latent=1)

model = vgae.SparseVGAE(model_configs)
device = torch.device('cpu')
optimizer = torch.optim.Adam(model.parameters(), lr=train_configs.lr)

In [ ]:
sample_idx = 0

sample_features = pd.read_csv(
    os.path.join(graph_path, filenames[sample_idx].split('.')[0]+'.csv'), index_col=[0]
).to_numpy()

# 
sample_graph = Gs[sample_idx]
sample_graph = sample_nodes(sample_graph, sample_ratio=0.01)
n_nodes = len(sample_graph)

x = torch.tensor(sample_features)
x = x.float()

# model = model.to(device)
# x = x.float().to(device)
# edge_index = edge_index.to(device)

In [ ]:
# Need (1). either re-generation subsampled graphs or (2). mini-batches for subsampling graphs w/ L1 loss computation
losses, nlls, sparse_losses, kls = model_train.train(model, sample_graph, sample_features, train_configs)

In [ ]:
# Check model reconstruction on adjacency matrix
A = nx.adjacency_matrix(sample_graph).toarray()

z = model_eval.eval(model, sample_graph, sample_features)
A_hat = z @ z.T

In [ ]:
# cleanup for retrain
del model, device, optimizer, train_configs, model_configs

In [ ]:
z = model_eval.eval(model, sample_graph, sample_features)
plt.figure(figsize=(4, 2))
plt.hist(z, bins=30)
plt.show()

In [ ]:
cyif_chans = [
    'DAPI',
    'GS 647', 
    'CYP3A4', 
    'ASS1 PE',
    'Pan CK',
    'CD31', 
    'CD45', 
    'CD56', 
    'CD68'
]

In [ ]:
img = imgs_rgb[sample_idx]
coords = coords_list[sample_idx]

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(21, 5))
ax1.imshow(img[1], cmap='magma', vmax=64)
ax1.axis('off')
ax1.set_title('GS (CV marker)')
ax2.imshow(img[2], cmap='magma', vmax=64)
ax2.axis('off')
ax2.set_title('CYP (PC marker)')
ax3.imshow(img[3], cmap='magma', vmax=64)
ax3.axis('off')
ax3.set_title('ASS1 (PP marker)')
qlow, qhigh = np.quantile(z, q=(0.1, 0.9))
im = ax4.scatter(coords[:, 0], coords[:, 1], s=1,
                 c=z, cmap='jet')
ax4.axis('off')
ax4.set_title('Learnt U')
plt.colorbar(im, ax=ax4)

plt.tight_layout()
plt.show()

In [ ]:
img = imgs_rgb[sample_idx]
coords = coords_list[sample_idx]
disp_network_embedding(img[sample_idx], sample_graph, embedding=z, alpha=0.25,
                       node_size=0.5, edge_width=0.5, figsize=(8, 8),
                       title='GS vs. Learnt U')

VGAE: PCs as baseline

In [ ]:
# Baseline: PCA??
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pc = pca.fit_transform(sample_features)

disp_network_embedding(img[3], sample_graph, embedding=pc[:, 0], alpha=0.25,
                       node_size=1, edge_width=0.2, figsize=(8, 8),
                       title='ASS vs. PC2')
                       

In [ ]:
plt.figure(figsize=(3, 1.5))
plt.hist(z, bins=60, edgecolor='white')
plt.title('Learnt U')
plt.show()

In [ ]:
plt.figure(figsize=(3, 1.5))
plt.hist(pc[:, 0], bins=60, edgecolor='white')
plt.title('PC1')
plt.show()

In [ ]:
from scipy.stats import pearsonr
plt.scatter(z.squeeze(), pc[:, 0], s=1)
plt.xlabel('Learnt U')
plt.ylabel('PC1')
plt.title('R={}'.format(
    np.round(pearsonr(z.squeeze(), pc[:, 0])[0], 4)
))

#### DESI + CyIF integration

- CyIF w/ Diff pooling as weak supervision for temperature
- Full DESI graph for temperature inference + 3D zonation assignments

-  **DEBUG**: current VGAE setup only passes `edge_index` without `edge_weight` (a.k.a no graph weight is encoded).... (done) <br> Debug reconstructed weights, sparsity (through **hierarchical designs**...)

In [ ]:
from cellpose.models import CellposeModel
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CellposeModel(
    gpu=device,
    model_type='nuclei',
    # pretrained_model=pretrained_model.value
)

In [ ]:
dapi = tifffile.imread('../data/desi/trimmed_cycif.tif')[0]
dapi = (dapi - dapi.min()) / (dapi.max() - dapi.min())
dapi.shape

In [ ]:
mask = model.eval(
    dapi,
    diameter=5,
    tile=True,
    channels=[0, 0],
    flow_threshold=4,
    min_size=30
)[0]

In [ ]:
coords = get_centroids(mask)
G = construct_graph(coords, k=10, r=50)
#G = sample_nodes(G_raw, k=30, sample_ratio=0.2)
#del G_raw, coords
# coords = get_coords_from_graph(G)  # Get coords from sampled graph

pickle.dump(G, open('../data/desi/cyif_graph.pkl', 'wb'))
gc.collect()


In [ ]:
coords = get_coords_from_graph(G)
img = tifffile.imread('../data/desi/trimmed_cycif.tif')[[0,1,2,6]].astype(np.float32)

img_normed = img.copy()
for i, chan in enumerate(img):
    img_normed[i] = (chan - chan.min()) / (chan.max() - chan.min())
feature_mat = utils.znorm(construct_feature_matrix(img_normed, coords, r=3))

feature_mat_df = pd.DataFrame(feature_mat, columns=['DAPI', 'ASS1', 'GS', 'CYP'])
feature_mat_df.to_csv('../data/desi/cyif_feature.csv', index=True)

Temperature inference on CyIF:

In [ ]:
from torch_geometric.nn import TopKPooling
from torch_geometric import utils as pyg_utils
from scipy.sparse import csr_matrix


def pooled_edge_attrs_to_graph(edge_index, edge_weight, 
                               G=None, perm=None,
                               to_nx=False):
    """
    Construct sparse adjacency matrix / nx.graph 
    from pooled (edge_index, edge_attrs)
    """
    if to_nx:
        assert G is not None and perm is not None, \
        "Requires original graph G & perm indices"

    adj_mat = pyg_utils.to_torch_coo_tensor(edge_index=edge_index, edge_attr=edge_weight.detach())
    
    if to_nx:
        adj_mat_scipy = csr_matrix((adj_mat.values(), adj_mat.indices()), shape=adj_mat.shape)
        G_new = nx.from_scipy_sparse_array(adj_mat_scipy)

        node_attrs = {n: G.nodes[k] for n, k in zip(G_new.nodes, perm)}
        nx.set_node_attributes(G_new, node_attrs)
        return adj_mat, G_new
    else:
        return adj_mat
    
    
def get_desi_features(desi_img, coords):
    n_cells = len(coords)
    n_features = len(desi_img)
    features = np.zeros((n_cells, n_features), dtype=np.float32)

    for j, chan in enumerate(desi_img):
        features[:, j] = chan[tuple(coords.T)]
    return features


def get_binned_features(features, nbins):
    features_binned = np.zeros((nbins, features.shape[1]))
    step = features.shape[0] // nbins
    for i, idx in enumerate(range(0, features.shape[0], step)):
        if i >= nbins:
            break
        features_binned[i] = features[idx:idx+step, :].mean(0)
    return features_binned


def disp_desi_gradients(sorted_features, labels, nbins=10):
    binned_features = get_binned_features(sorted_features, nbins=nbins)  # (coord x feature)
    g = sns.clustermap(binned_features.T, 
                       row_cluster=True, col_cluster=False, 
                       cmap='coolwarm', figsize=(8, 8))

    ax = g.ax_heatmap
    ax.set_xlabel('PV -> CV', fontsize=15)
    ax.set_ylabel('DESI chans', fontsize=15)
    step = np.round(len(labels)/len(ax.get_yticklabels())).astype(np.int8)
    ax.set_yticklabels(labels[::step])
    ax.set_title('DESI gradients (# bins={})'.format(nbins), fontsize=20)

    plt.show()  


In [ ]:
img = tifffile.imread('../data/desi/trimmed_cycif.tif')[[0,1,2,6]].astype(np.float32)
G = pickle.load(open('../data/desi/cyif_graph.pkl', 'rb'))
coords = get_coords_from_graph(G)
feature_mat = pd.read_csv('../data/desi/cyif_feature.csv', index_col=[0]).values

graph_data = pyg_utils.from_networkx(G)
edge_index, edge_weight = graph_data.edge_index, graph_data.weight

edge_index, edge_weight = utils.nx_to_edge_attrs(G)
x = torch.tensor(feature_mat, dtype=torch.float32)
# adj_mat = pyg_utils.to_torch_csr_tensor(edge_index=edge_index, edge_attr=edge_weight)

In [ ]:
disp_network(img[0], G, figsize=(12, 12), title='Original G')

In [ ]:
topk_pool = TopKPooling(4, ratio=0.5, nonlinearity='sigmoid')

x, ei, ew, _, perm, _ = topk_pool(x=x, edge_index=edge_index, edge_attr=edge_weight)
perm = perm.detach().numpy()
adj_matrix_pooled, G_pooled = pooled_edge_attrs_to_graph(ei,
                                                         ew,
                                                         G=G,
                                                         perm=perm,
                                                         to_nx=True)

disp_network(img[0], G_pooled, figsize=(12, 12), title='Pooled G (topK)')


In [ ]:
# Hierarchical pooling
x, ei, ew, _, perm, _ = topk_pool(x=x, edge_index=ei, edge_attr=ew)
perm = perm.detach().numpy()
adj_matrix_pooled, G_pooled = pooled_edge_attrs_to_graph(ei,
                                                         ew,
                                                         G=G_pooled,
                                                         perm=perm,
                                                         to_nx=True)

disp_network(img[0], G_pooled, figsize=(12, 12), title='Pooled G (topK)')

In [ ]:
adj_matrix_pooled.shape

In [ ]:
plt.imshow(adj_matrix_pooled.to_dense().numpy()[:256, :256])

---

In [ ]:
desi_raw = tifffile.imread('../data/desi/Desi_neg_7.tif')
desi = desi_raw.transpose(0,2,1)[:, 82:222, 60:200]  # Trim & rotate/flip & resize to CyIF patch

In [ ]:
# Model configs & initialization
n_nodes = len(G)
lr = 1e-3
n_epochs = 50

train_configs = configs.set_train_configs(data_path=None, 
                                          n_epochs=n_epochs, 
                                          lr=lr)

model_configs = configs.set_model_configs(beta=0,
                                          c_in=4,
                                          c_hidden=2,
                                          c_latent=1)

model = vgae.SparseVGAE(model_configs)
device = torch.device('cpu')
optimizer = torch.optim.Adam(model.parameters(), lr=train_configs.lr)

In [ ]:
losses, nlls, sparse_losses, kls = model_train.train(model, G, feature_mat, train_configs)

In [ ]:
del model, model_configs, train_configs, optimizer

In [ ]:
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 14))

ax1.plot(np.arange(n_epochs), losses)
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total Loss')

ax2.plot(np.arange(n_epochs), nlls)
ax2.set_xlabel('Epochs')
ax2.set_ylabel('NLLs')

ax3.plot(np.arange(n_epochs), sparse_losses)
ax3.set_xlabel('Epochs')
ax3.set_ylabel('L1-regularization')

ax4.plot(np.arange(n_epochs), kls)
ax4.set_xlabel('Epochs')
ax4.set_ylabel('KLs')

plt.tight_layout()
plt.show()

In [ ]:
z = model_eval.eval(model, G, feature_mat)
edge_index, _ = utils.nx_to_edge_attrs(G)
x = torch.tensor(feature_mat).float()

# Graph reconstruction checks
A = nx.adjacency_matrix(G).toarray()
A_hat = model.decoder.forward_all(torch.tensor(z), edge_index, x).detach().cpu().numpy()


In [ ]:
disp_network_embedding(img[2], G, embedding=z, alpha=0.5, 
                       node_size=2, edge_width=1, fontsize=64,
                       figsize=(15, 15), title='GS vs. Learnt U')

In [ ]:
disp_network_embedding(img[3], G, embedding=z, alpha=0.5, 
                       node_size=2, edge_width=1, fontsize=40,
                       figsize=(15, 15), title='CYP vs. Learnt U')

In [ ]:
disp_network_embedding(img[1], G, embedding=z, alpha=0.5, 
                       node_size=2, edge_width=1, fontsize=40,
                       figsize=(15, 15), title='ASS1 vs. Learnt U')

In [ ]:
# Save graph with inferred temperature
attrs = {node: u for (node, u) in zip(G.nodes, z.flatten())}
nx.set_node_attributes(G, attrs, name='u')
pickle.dump(G, open('../data/desi/cyif_graph.pkl', 'wb'))

Integration: Collapse CyIF graph into DESI grids

In [ ]:
# Load saved CyIF graph w/ inferred temperature
cyif_dapi = tifffile.imread('../data/desi/trimmed_cycif.tif')[0]
cyif_size = cyif_dapi.shape
G = pickle.load(open('../data/desi/cyif_graph.pkl', 'rb'))

# Load DESI features
desi_raw = tifffile.imread('../data/desi/Desi_neg_7.tif')
desi = desi_raw.transpose(0,2,1)[:, 82:222, 60:200]  # Trim & rotate/flip & resize to CyIF patch
desi_size = desi.shape[1:]

# Load DESI m/z labels:
ifile = open('../data/desi/Desi_neg_7.tif', 'rb')
ion_labels = IO.load_ome_labels(ifile, '../data/desi/Desi_neg_7.tif')
ifile.close()

In [ ]:
# Create bags of nodes by mixing CyIF coords into DESI pixels
scaleY, scaleX = cyif_size[0] / desi_size[0], cyif_size[1] / desi_size[1]

yy, xx = np.meshgrid(np.arange(desi_size[0]), np.arange(desi_size[1]))
yy = yy.flatten()
xx = xx.flatten()
desi_coord_dict = {(yi, xi): [] for xi, yi in zip(xx.flatten(), yy.flatten())}  # IJ-based keys

for n in G.nodes:
    xi, yi = G.nodes[n]['pos']
    u = G.nodes[n]['u']
    yi = min(np.round(yi/scaleY).astype(np.uint32), desi_size[0]-1)
    xi = min(np.round(xi/scaleX).astype(np.uint32), desi_size[1]-1)
    desi_coord_dict[(yi, xi)].append(u)


In [ ]:
desi_temperature = np.ones(desi_size[0]*desi_size[1]) * -np.inf
for i, (k, v) in enumerate(desi_coord_dict.items()):
    if len(v) > 0:
        desi_temperature[i] = np.mean(v)

In [ ]:
plt.figure(figsize=(8, 8))
im = plt.imshow(desi_temperature.reshape(desi_size[0], desi_size[1]).T, cmap='coolwarm')  # Converting XY coords to IJ coords
plt.colorbar(im)

In [ ]:
# For all DESI pixels with non-empty colors, sort feature matrix by temperatures
desi_coords = []
for k, v in desi_coord_dict.items():
    if len(v) > 0:
        desi_coords.append(list(k))
desi_coords = np.array(desi_coords)

In [ ]:
desi_features = get_desi_features(desi, desi_coords)
desi_features_sorted = desi_features[np.argsort(desi_temperature[desi_temperature != -np.inf])]

In [ ]:
disp_desi_gradients(desi_features_sorted, ion_labels, nbins=10)

In [ ]:
disp_desi_gradients(desi_features_sorted, ion_labels, nbins=20)

In [ ]:
disp_desi_gradients(desi_features_sorted, ion_labels, nbins=100)

### VGAE w/ Hierarchical representations

- Dim-reduction on nodes (Diffpools)
- Dim-reduction on features (`z`: representation, `u`: interpretability)<br> $z \in \mathbb{R^{N' \times 10}}$, $u \in \mathbb{R}^{N' \times 1}$
- Reconstruction w/ $\sigma(z \cdot z^T)$ for graph reconstruction, regularize $u$ w/ CyCIF graph integration

In [ ]:
import vgae, utils
import model_train, model_eval, configs

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.data import ClusterData



In [ ]:
# Load DESI normed image

# desi_raw = tifffile.imread('../data/desi/Desi_neg_7.tif').astype(np.float32)
# desi_img = desi_raw.transpose(0,2,1)[:, 82:222, 60:200]  # Trim & rotate/flip & resize to CyIF patch
# desi_normed = np.zeros_like(desi_img)
# for i in range(desi_img.shape[0]):
#     desi_normed[i] = (desi_img[i] - desi_img[i].min()) / (desi_img[i].max() - desi_img[i].min())

# tifffile.imwrite('../data/desi/Desi_neg_7_aligned.tif', desi_normed)

desi_img = tifffile.imread('../data/desi/Desi_neg_7_aligned.tif')

yy, xx = np.meshgrid(np.arange(desi_img.shape[-2]), np.arange(desi_img.shape[-1]))
desi_coords = np.array([yy.flatten(), xx.flatten()]).T
desi_feature_mat = construct_feature_matrix(desi_img, desi_coords, r=1)

# Convert desi_image to [C, X, Y] to
# extract features by row (as the inner ptr for "flatten")
nchans = desi_img.shape[0]
desi_feature_mat = desi_img.transpose(0,2,1).reshape(nchans, -1).T  
G_desi = construct_graph(desi_coords, k=4, r=1)  # 4-connected lattice graph
edge_index, edge_weight = utils.nx_to_edge_attrs(G_desi)

In [ ]:
disp_network(desi_img[7], G_desi, figsize=(12, 12), node_size=1, edge_width=0.5)

Create sub-graphs with mini-batches

In [ ]:
def create_ims_loader(G, feature_mat, 
                      n_batches=1,
                      n_subgraphs=1,
                      is_weighted=False):
    """
    Convert networkx graphs to minibatches of `(edge_index, edge_attr)` subgraphs
    """
    data = pyg_utils.from_networkx(G)
    if not is_weighted:
        data.weight = None
    data.x = feature_mat.float() if isinstance(feature_mat, torch.Tensor) \
         else torch.tensor(desi_feature_mat).float() 
    
    ims_data = ClusterData(data, num_parts=n_subgraphs) if n_subgraphs > 1 else data
    ims_loader = DataLoader(ims_data, batch_size=n_batches)

    return ims_loader


In [ ]:
# Check graph partition
ims_loader = create_ims_loader(G_desi, desi_feature_mat, n_subgraphs=8)
subgraphs = []
for subgraph_loader in ims_loader:
    subgraph = pyg_utils.to_networkx(subgraph_loader, node_attrs=['pos'], to_undirected=True)
    subgraphs.append(subgraph)
    

In [ ]:
# DEBUG model
from importlib import reload
reload(vgae)
reload(utils)
reload(model_train)
reload(model_eval)
reload(configs)

In [ ]:
# Try training on subgraph
# Model configs & initialization
lr = 1e-3
n_epochs = 50

train_configs = configs.set_train_configs(data_path=None, 
                                          n_epochs=n_epochs, 
             
                                          lr=lr)

model_configs = configs.set_model_configs(beta=1e-2,
                                          c_in=nchans,
                                          c_hidden=64,
                                          c_latent=1,
                                          sparse_prior=0.1,
                                          pz_std=1)

model = vgae.SparseVGAE(model_configs)
device = torch.device('cpu')
optimizer = torch.optim.Adam(model.parameters(), lr=train_configs.lr)

In [ ]:
losses, nlls, sparse_losses, kls = model_train.train(model, ims_loader, train_configs)

In [ ]:
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 14))

ax1.plot(np.arange(n_epochs*len(ims_loader)), losses)
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total Loss')

ax2.plot(np.arange(n_epochs*len(ims_loader)), nlls)
ax2.set_xlabel('Epochs')
ax2.set_ylabel('NLLs')

ax3.plot(np.arange(n_epochs*len(ims_loader)), sparse_losses)
ax3.set_xlabel('Epochs')
ax3.set_ylabel('Smooth regularization')

ax4.plot(np.arange(n_epochs*len(ims_loader)), kls)
ax4.set_xlabel('Epochs')
ax4.set_ylabel('KLs')

plt.tight_layout()
plt.show()

In [ ]:
# Graph reconstruction checks 

# subgraph
# A = nx.adjacency_matrix(subgraph).toarray()
# latent = model_eval.eval(model, subgraph, graph_data.x)
# reconst = model.decoder(latent)

# full graph
A = nx.adjacency_matrix(G_desi).toarray()
latent = model_eval.eval(model, G_desi, desi_feature_mat)
reconst = model.decoder(latent)

z = latent.z.detach().cpu().numpy()
u = latent.u.detach().cpu().numpy()
A_hat = reconst.A_hat.detach().cpu().numpy()

In [ ]:
plt.imshow(A[:512, :512], cmap='magma')

In [ ]:
plt.imshow(A_hat[:512, :512], cmap='magma')

In [ ]:
disp_network_embedding(desi_img.mean(0), G_desi, embedding=u.flatten(), alpha=0.5, 
                       node_size=5, edge_width=0.5, fontsize=64,
                       figsize=(15, 15), title='DESI vs. Learnt U')

Modality integration **Using CyIF as priors to regularize u**:

- Vanilla design: (1). distance transformation to closest, **binarized** GS signal; (2). Graph-based heat diffusion???

In [ ]:
from skimage.filters import threshold_otsu
from skimage.morphology import binary_erosion
from skimage.transform import resize
from scipy import ndimage as ndi

In [ ]:
def get_gs_dist(gs):
    assert 0 <= gs.min() <= gs.max() <= 1, \
        "Single-channel intensity should be normalized to [0, 1]"

    # fp = np.array([[0, 1, 0],
    #                [1, 1, 1],
    #                [0, 1, 0]])
    thresh = threshold_otsu(gs)
    gs_bin = (gs > thresh).astype(np.uint8)
    gs_bin = binary_erosion(gs_bin, np.ones((7, 7)))
    gs_dist = ndi.distance_transform_edt(1-gs_bin)
    return gs_dist


In [ ]:
cyif_img = tifffile.imread('../data/desi/trimmed_cycif.tif')[[2, 6, 1]].astype(np.float32)  # Only load GS, CYP & ASS

for i in range(len(cyif_img)):
    cyif_img[i] = (cyif_img[i]-cyif_img[i].min()) / (cyif_img[i].max()-cyif_img[i].min())
# Intenstity: [0, 1]

In [ ]:
fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(21, 5))
ax1.imshow(cyif_img[0], cmap='magma')
ax1.axis('off')
ax1.set_title('GS')
ax2.imshow(cyif_img[1], cmap='magma')
ax2.axis('off')
ax2.set_title('CYP')
ax3.imshow(cyif_img[2], cmap='magma')
ax3.axis('off')
ax3.set_title('ASS')

empty_chan = np.ones_like(cyif_img[0]) 
empty_chan -= cyif_img.max(0)
ax4.imshow(empty_chan, cmap='magma')
ax4.set_title('1-MIP')

plt.tight_layout()
plt.show()

In [ ]:
u_prior = resize(
    get_gs_dist(cyif_img[0]),
    output_shape=desi_img.shape[-2:],
    preserve_range=True
)

u_prior = u_prior.max() - u_prior  # Flip sign
u_prior = (u_prior - u_prior.min()) / (u_prior.max() - u_prior.min())  # Norm to [0, 1]

In [ ]:
plt.imshow(u_prior, cmap='magma')
plt.colorbar()
plt.show()

In [ ]:
tifffile.imwrite('../data/desi/cyif_prior.tif', u_prior)

---